# Greece 2023 -- Zero-Shot Geographic Generalization

**Environment**: Colab A100 (GPU required)

Model trained on Corrientes, Argentina (subtropical savanna, Dec 2021-Feb 2022).
Evaluated zero-shot on Alexandroupolis, Greece (Mediterranean shrubland, August 2023).
No Greece annotations were used at any point.

---

| Cells | Section | Description |
|---|---|---|
| 1 | Setup | Install, Drive mount |
| 2-3 | Imports, Paths, Architecture | Load model v1.5 |
| 4 | Dataset | Greece 2023 patches |
| 5-6 | Inference | ZS metrics + curves |
| 7 | Visualisation | Predictions on fire patches |
| 8 | Summary figure | Cross-biome generalization overview |

In [ ]:
# [01] Environment setup
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running in Google Colab -- Drive mounted.')

    needs = False
    try:
        import numpy as np
        assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
        from terratorch.registry import BACKBONE_REGISTRY
        import segmentation_models_pytorch
        print(f'OK -- numpy={np.__version__}, terratorch ready.')
    except Exception as e:
        needs = True
        print(f'Installing ({e})...')

    if needs:
        import subprocess, sys, os
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'numpy==2.0.2', '--force-reinstall'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'terratorch', 'einops', 'timm',
                        'segmentation-models-pytorch'], check=True)
        os.kill(os.getpid(), 9)

except ImportError:
    IN_COLAB = False
    print('Running locally.')
    import numpy as np

In [ ]:
# [02] Imports and paths
import os, random, warnings, subprocess
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                              precision_recall_curve, roc_curve)
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('..').resolve()  # repo root

CKPT_V15 = BASE / 'models' / 'best_prithvi_burn_scar_wildfire.pth'
RESULTS  = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

# Copy patches to /content/ to avoid Drive I/O bottleneck
if IN_COLAB:
    LOCAL_GR = Path('/content/greece_patches')
    LOCAL_GR.mkdir(exist_ok=True)
    for subdir in ('images', 'masks_dnbr'):
        dst = LOCAL_GR / subdir
        src = BASE / 'data' / 'greece' / 'patches' / subdir
        if not dst.exists():
            print(f'Copying {subdir} from Drive...')
            subprocess.run(['cp', '-r', str(src), str(LOCAL_GR)], check=True)
        else:
            print(f'Already cached: {subdir}')
    GR_IMG_DIR  = LOCAL_GR / 'images'
    GR_MASK_DIR = LOCAL_GR / 'masks_dnbr'
else:
    GR_IMG_DIR  = BASE / 'data' / 'greece' / 'patches' / 'images'
    GR_MASK_DIR = BASE / 'data' / 'greece' / 'patches' / 'masks_dnbr'

img_paths  = sorted(GR_IMG_DIR.glob('*.npy'))
mask_paths = sorted(GR_MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths) and len(img_paths) > 0, \
    f'Patch mismatch or empty dir. imgs={len(img_paths)} masks={len(mask_paths)}'
print(f'Greece patches: {len(img_paths)}')

In [ ]:
# [03] Architecture -- identical to notebook 04 v1.5 (must match checkpoint)
from terratorch.registry import BACKBONE_REGISTRY

PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
BAND_IDX     = [0, 1, 2, 4, 5, 6]   # B02, B03, B04, B8A, B11, B12
IMG_SIZE     = 224
T            = (256 - IMG_SIZE) // 2
THRESHOLD    = 0.525


class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=768, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([2, 5, 8, 11])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100')
        self.neck     = MultiScaleNeck(n_side=14, d_model=768, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))


model = PrithviSegModelV2().to(DEVICE)
state = torch.load(CKPT_V15, map_location=DEVICE)
model.load_state_dict(state)
model.eval()
print(f'Model loaded: {CKPT_V15.name}')
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters : {n_params:.1f}M')

In [ ]:
# [04] Greece 2023 dataset
class GreeceDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx],  mmap_mode='r').astype(np.float32)
        mask = np.load(self.mask_paths[idx], mmap_mode='r').astype(np.float32)
        img  = img[BAND_IDX] / 10000.0
        img  = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask = mask[T:T+IMG_SIZE, T:T+IMG_SIZE]
        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)


ds     = GreeceDataset(img_paths, mask_paths)
loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths, desc='Checking masks')
], dtype=bool)

print(f'Patches total    : {len(img_paths)}')
print(f'  Fire patches   : {fire_flags.sum()}')
print(f'  Background     : {(~fire_flags).sum()}')
print(f'  Positive rate  : {fire_flags.mean()*100:.1f}%')

In [ ]:
# [05] Zero-shot inference
all_probs, all_targets = [], []
patch_probs_list, patch_masks_list = [], []

print('Running zero-shot inference on Greece 2023...')
with torch.no_grad():
    for imgs, masks in tqdm(loader):
        probs    = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
        masks_np = masks.squeeze(1).cpu().numpy()
        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))
        for b in range(len(imgs)):
            patch_probs_list.append(probs[b])
            patch_masks_list.append(masks_np[b])

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
tp = int(((all_preds==1)&(all_targets==1)).sum())
fp = int(((all_preds==1)&(all_targets==0)).sum())
fn = int(((all_preds==0)&(all_targets==1)).sum())
iou = tp / (tp + fp + fn + 1e-6)
try:
    auc = roc_auc_score(all_targets, all_probs)
except Exception:
    auc = float('nan')

print()
print('=' * 55)
print('  Zero-Shot Results -- Greece 2023 (Alexandroupolis)')
print('=' * 55)
print(f'  IoU       : {iou:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  F1        : {f1:.4f}')
print(f'  AUC-ROC   : {auc:.4f}')
print(f'  TP: {tp}   FP: {fp}   FN: {fn}')
print('=' * 55)

In [ ]:
# [06] Precision-Recall and ROC curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')

prec_c, rec_c, _ = precision_recall_curve(all_targets, all_probs)
axes[0].plot(rec_c, prec_c, color='#C0392B', lw=2)
axes[0].scatter([recall], [precision], color='black', zorder=5, s=80,
                label=f'thr={THRESHOLD}  F1={f1:.3f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall -- Greece 2023 ZS')
axes[0].legend(frameon=False); axes[0].grid(alpha=0.3)

if not np.isnan(auc):
    fpr_c, tpr_c, _ = roc_curve(all_targets, all_probs)
    axes[1].plot(fpr_c, tpr_c, color='#2980B9', lw=2, label=f'AUC={auc:.3f}')
    axes[1].plot([0, 1], [0, 1], '--', color='gray', lw=1)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC -- Greece 2023 ZS')
    axes[1].legend(frameon=False); axes[1].grid(alpha=0.3)

plt.suptitle('Zero-Shot Geographic Generalization: Corrientes to Greece', fontsize=13)
plt.tight_layout()
out = RESULTS / 'greece_zs_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# [07] Visualisation -- best and worst fire patch predictions
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

patch_ious = np.array([
    float(((p > THRESHOLD).astype(np.float32) * m).sum() /
          (((p > THRESHOLD).astype(np.float32) + m).clip(0, 1).sum() + 1e-6))
    for p, m in zip(patch_probs_list, patch_masks_list)
])

fire_idx  = [i for i, f in enumerate(fire_flags) if f]
best_idx  = sorted(fire_idx, key=lambda i: patch_ious[i], reverse=True)[:4]
worst_idx = sorted(fire_idx, key=lambda i: patch_ious[i])[:4]


def make_grid(indices, title, save_name):
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n), facecolor='white')
    for ri, vi in enumerate(indices):
        raw  = np.load(img_paths[vi], mmap_mode='r').astype(np.float32)
        rgb  = np.stack([norm_band(raw[2, T:T+IMG_SIZE, T:T+IMG_SIZE]),
                         norm_band(raw[1, T:T+IMG_SIZE, T:T+IMG_SIZE]),
                         norm_band(raw[0, T:T+IMG_SIZE, T:T+IMG_SIZE])], axis=-1)
        prob = patch_probs_list[vi]
        mask = patch_masks_list[vi]
        pred = (prob > THRESHOLD).astype(np.float32)
        err  = np.zeros((*mask.shape, 3), dtype=np.float32)
        err[((pred==1)&(mask==1))] = [0, 1, 0]
        err[((pred==1)&(mask==0))] = [1, 0, 0]
        err[((pred==0)&(mask==1))] = [0, 0, 1]
        row_axes = axes[ri] if n > 1 else axes
        for ci, (img_data, cmap, ttl) in enumerate([
            (rgb, None, f'RGB  IoU={patch_ious[vi]:.3f}'),
            (mask, 'Reds', 'Ground truth (dNBR)'),
            (prob, 'RdYlGn_r', 'Predicted probability'),
            (err, None, 'TP/FP/FN'),
        ]):
            row_axes[ci].imshow(img_data, cmap=cmap, vmin=0, vmax=1)
            if ri == 0: row_axes[ci].set_title(ttl, fontsize=9)
            row_axes[ci].axis('off')
    last = axes[-1] if n > 1 else axes
    for label, color in [('TP', 'green'), ('FP', 'red'), ('FN', 'blue')]:
        last[-1].add_patch(mpatches.Patch(color=color, label=label))
    last[-1].legend(loc='lower right', fontsize=8, frameon=False)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    out = RESULTS / save_name
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')


make_grid(best_idx,  'Greece ZS: Best Predictions',  'greece_zs_best.png')
make_grid(worst_idx, 'Greece ZS: Worst Predictions', 'greece_zs_worst.png')

In [ ]:
# [08] Cross-biome generalization summary figure
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')

conditions = ['Corrientes\n(train/val)', 'Cordoba\n(ZS, T=1)', 'Greece\n(ZS, T=1)']
ious = [0.538, 0.1149, iou]
colors = ['#2E86AB', '#E84855', '#C0392B']
x = np.arange(len(conditions))
bars = axes[0].bar(x, ious, color=colors, width=0.55, edgecolor='none')
for bar, v in zip(bars, ious):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(conditions, fontsize=9)
axes[0].set_ylim(0, 0.75); axes[0].set_ylabel('IoU')
axes[0].set_title('IoU -- Cross-Biome Generalization')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

zs_conditions = ['Cordoba ZS', 'Greece ZS']
zs_aucs = [0.7383, auc]
zs_recalls = [0.2089, recall]
x2 = np.arange(2)
w  = 0.35
b1 = axes[1].bar(x2 - w/2, zs_aucs,   w, label='AUC-ROC',
                 color=['#2980B9', '#1A5276'], edgecolor='none')
b2 = axes[1].bar(x2 + w/2, zs_recalls, w, label='Recall',
                 color=['#85C1E9', '#5DADE2'], edgecolor='none')
for bars_grp in (b1, b2):
    for bar in bars_grp:
        h = bar.get_height()
        if not np.isnan(h):
            axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.01,
                         f'{h:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(x2); axes[1].set_xticklabels(zs_conditions, fontsize=9)
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('Score')
axes[1].set_title('AUC-ROC and Recall -- ZS Comparison')
axes[1].legend(fontsize=9, frameon=False)
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle(
    'Prithvi-100M: trained on Corrientes (subtropical savanna), evaluated on unseen biomes',
    fontsize=11)
plt.tight_layout()
out = RESULTS / 'greece_cross_biome_summary.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# [09] Portfolio summary
print('=' * 65)
print('  PORTFOLIO SUMMARY -- Cross-Biome Zero-Shot Generalization')
print('=' * 65)
print('  Backbone : Prithvi-EO-1.0-100M (IBM/NASA, frozen)')
print('  Decoder  : trained on Corrientes, Argentina (Dec 2021 fire)')
print('  Biome    : subtropical savanna / wetlands (humid subtropical)')
print('-' * 65)
print('  TRAIN (Corrientes, Argentina):')
print('    Location : -59.5W/-29.0S to -56.0W/-26.5N')
print('    Val IoU  : 0.538 (threshold-optimized)')
print('-' * 65)
print('  ZS TEST 1 -- Cordoba, Argentina (Argentine Monte / highland scrub):')
print(f'    IoU: 0.1149   AUC-ROC: 0.7383   Recall: 0.2089')
print('-' * 65)
print(f'  ZS TEST 2 -- Alexandroupolis, Greece (Mediterranean shrubland):')
print(f'    IoU: {iou:.4f}   AUC-ROC: {auc:.4f}   Recall: {recall:.4f}')
print(f'    Precision: {precision:.4f}   F1: {f1:.4f}')
print('    Zero Greece annotations used at any stage.')
print('=' * 65)